<a href="https://colab.research.google.com/github/drawcodeboy/Cat_n_Dog_Classification/blob/main/cat_n_dog_modeling_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.model_selection import train_test_split

In [3]:
# Modeling
model = keras.Sequential()
model.add(keras.layers.Conv2D(32, kernel_size=3, activation='relu', 
                              kernel_regularizer=tf.keras.regularizers.l2(0.2), 
                              padding = 'same', input_shape=(224, 224, 3)))
model.add(keras.layers.MaxPooling2D(2))
model.add(keras.layers.Conv2D(64, kernel_size=3, activation='relu', 
                              kernel_regularizer=tf.keras.regularizers.l2(0.02), 
                              padding = 'same'))
model.add(keras.layers.MaxPooling2D(2))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dropout(0.3))
model.add(keras.layers.Dense(50, activation='relu'))
model.add(keras.layers.Dense(1, activation='sigmoid'))

In [4]:
train_path = '/content/drive/MyDrive/cat_n_dog/data_scaled_numpy/training_set_scaled'
train_path_list = []

for dirname, _, filenames in os.walk(train_path):
    for filename in filenames:
        if(filename.endswith('npy')):
            train_set_part_path = os.path.join(dirname, filename)
            train_path_list.append(train_set_part_path)

train_path_list.sort()

In [5]:
train_target_path = '/content/drive/MyDrive/cat_n_dog/data_scaled_numpy/train_target_part'
train_target_path_list = []

for dirname, _, filenames in os.walk(train_target_path):
    for filename in filenames:
        if(filename.endswith('npy')):
            train_target_part_path = os.path.join(dirname, filename)
            train_target_path_list.append(train_target_part_path)
        
train_target_path_list.sort()

In [6]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics='accuracy')

early_stopping_cb = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

In [7]:
def load_data(x_path, y_path):
    x = np.load(x_path, allow_pickle=True)
    y = np.load(y_path, allow_pickle=True)
    return (x, y)

In [8]:
def model_fit(train_set_, train_target_):
    # train_set, validation_set 분리
    x_train, x_val, y_train, y_val = train_test_split(
        train_set_, train_target_, test_size=0.2, random_state=42
    )

    # Training
    hist = model.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val),
               callbacks=[early_stopping_cb], batch_size = 64)

    return hist

In [ ]:
history = []

for x_, y_ in zip(train_path_list, train_target_path_list):
    X, Y = load_data(x_, y_)
    X = X / 255.0
    history.append(model_fit(X, Y))
    del X
    del Y

Epoch 1/20
25/25 [==============================] - 98s 4s/step - loss: 3.2793 - accuracy: 0.5081 - val_loss: 2.1084 - val_accuracy: 0.4450
Epoch 2/20
25/25 [==============================] - 93s 4s/step - loss: 1.9551 - accuracy: 0.5294 - val_loss: 1.7987 - val_accuracy: 0.5500
Epoch 3/20
19/25 [=====================>........] - ETA: 20s - loss: 1.7379 - accuracy: 0.5551

In [ ]:
def show_graph(history_):
    accuracy = history_.history['accuracy']
    val_accuracy = history_.history['val_accuracy']
    loss = history_.history['loss']
    val_loss = history_.history['val_loss']

    epochs = range(1, len(loss) + 1)

    plt.figure(figsize=(10, 2))

    plt.subplot(121)
    plt.ylim(0, 1.1)
    plt.subplots_adjust(top=2)
    plt.plot(epochs, accuracy, 'ro', label='Training accuracy')
    plt.plot(epochs, val_accuracy, 'r', label='Validation accuracy')
    plt.title('Trainging and validation accuracy and loss')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy and Loss')

    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
              fancybox=True, shadow=True, ncol=5)
#     plt.legend(bbox_to_anchor=(1, -0.1))

    plt.subplot(122)
    plt.plot(epochs, loss, 'bo', label='Training loss')
    plt.plot(epochs, val_loss, 'b', label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
          fancybox=True, shadow=True, ncol=5)
#     plt.legend(bbox_to_anchor=(1, 0))

    plt.show()

reference
https://chealin93.tistory.com/69

In [ ]:
for x in history:
    show_graph(x)

In [ ]:
# Load Test Set

test_set = np.load('/content/drive/MyDrive/cat_n_dog/data_scaled_numpy/test_set_scaled/test_scaled.npy', allow_pickle=True)
test_target = np.load('/content/drive/MyDrive/cat_n_dog/data_numpy/test_target.npy', allow_pickle=True)

In [ ]:
test_set_scaled = test_set / 255.0

model.evaluate(test_set_scaled, test_target)

In [ ]:
sample = test_set[58] / 255.0
sample = sample.reshape(1, 224, 224, 1)
print(sample.shape)
plt.imshow(test_set[58])
plt.show()

In [ ]:
def print_predict(sample_):
    result = model.predict(sample)
    result = result[0][0]
    

    print('RESULT: ', end='')
    if(result < 0.5):
        print('Cat')
    elif(result > 0.5):
        print('Dog')
    else:
        print('Can\'t know')

In [ ]:
print_predict(sample)